# Analysis 03: Painting Options Exploration

In this notebook, we explore the impact of different painting methods (NGP vs BILINEAR) and the effect of up-painting (painting to 2048 and upgrading to 1024) on the angular power spectra of the shells.

The simulations were run with a mesh size of 2048x2048x2048 and a box size of 6000 Mpc/h.

In [1]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['JAX_PLATFORM_NAME'] = 'cpu'

import glob
import jax.numpy as jnp
import jax_fli as jfli
import datasets
import numpy as np
import matplotlib.pyplot as plt
from utils import set_jcap_style

set_jcap_style()

data_dirs = {
    'NGP': '../results/03-painting_options/COPY/cosmology_runs_NGP_NOUPPAINT/',
    'NGP (Up-paint)': '../results/03-painting_options/COPY/cosmology_runs_NGP_UPPAINT/',
    'BILINEAR': '../results/03-painting_options/COPY/cosmology_runs_BILINEAR_NOUPPAINT/',
    'BILINEAR (Up-paint)': '../results/03-painting_options/COPY/cosmology_runs_BILINEAR_UPPAINT/'
}

parquet_files = {}
for label, d in data_dirs.items():
    files = glob.glob(os.path.join(d, '*.parquet'))
    if files:
        parquet_files[label] = files[0]
        print(f"{label}: {os.path.basename(files[0])}")


In [2]:
from tqdm import tqdm

spectra_by_label = {}   # label -> angular_cl (all shells)
cosmo_by_label = {}     # label -> cosmology
centers_by_label = {}   # label -> comoving_centers
field_shape = None
density_width = None

for label, fpath in tqdm(parquet_files.items()):
    ds = datasets.load_dataset('parquet', data_files=[fpath], split='train').with_format('numpy')
    item = ds[0]
    catalog = jfli.io.Catalog.from_dataset(item)
    field = catalog.field[0]
    cosmology = catalog.cosmology[0]

    if field_shape is None: field_shape = field.shape
    if density_width is None: density_width = field.density_width[0]

    overdensity_lc = (field / field.mean(axis=1, keepdims=True)) - 1.
    angular_cl = overdensity_lc.angular_cl(method='healpy')

    spectra_by_label[label] = angular_cl
    cosmo_by_label[label] = cosmology
    centers_by_label[label] = field.comoving_centers


0it [00:00, ?it/s]


In [3]:
import jax_cosmo as jc

# Pick a specific shell to plot (e.g., the last one)
SHELL_IDX = -1
ref_label = list(parquet_files.keys())[0]
n_shells = field_shape[0]
shell_abs = SHELL_IDX % n_shells

print(f"Computing theory Cl for shell {SHELL_IDX} (absolute index {shell_abs})...")

cosmo = cosmo_by_label[ref_label]
comoving_centers = centers_by_label[ref_label]

ref_nside = 1024
LMAX = 3 * ref_nside
ells = jnp.arange(LMAX)

r_near = comoving_centers[shell_abs] - density_width / 2
r_far  = comoving_centers[shell_abs] + density_width / 2
z_near, z_far = jc.utils.a2z(
    jc.background.a_of_chi(cosmo, jnp.array([r_near, r_far]))
)
nz = jfli.tophat_z(float(z_near), float(z_far), gals_per_arcmin2=1.0)
theory_cl = jfli.compute_theory_cl(
    cosmo,
    ell=ells,
    z_source=[nz],
    probe_type='number_counts',
    nonlinear_fn=jc.power.halofit,
    cross=False,
)

print("Done.")


IndexError: list index out of range

In [ ]:
fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(8, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

ell_theory = jnp.asarray(theory_cl.wavenumber)
cl_theory_arr = jnp.asarray(theory_cl.spectra)

# Top plot: Spectra
ax_top.plot(ell_theory, cl_theory_arr, color='black', linestyle='--', label='Theory (Halofit)')

colors = plt.cm.Set1(np.linspace(0, 1, len(spectra_by_label)))

for i, (label, angular_cl) in enumerate(spectra_by_label.items()):
    cl_sim = angular_cl[SHELL_IDX]
    ell_sim = cl_sim.wavenumber
    cl_sim_vals = cl_sim.spectra
    
    ax_top.loglog(ell_sim, cl_sim_vals, label=label, color=colors[i], alpha=0.8)
    
    # Bottom plot: Ratio
    # Interpolate theory to sim ells for ratio
    cl_theory_interp = jnp.interp(ell_sim, ell_theory, cl_theory_arr)
    ratio = cl_sim_vals / cl_theory_interp
    
    ax_bot.semilogx(ell_sim, ratio, color=colors[i], alpha=0.8)

ax_top.set_ylabel(r"$C_\ell$")
ax_top.set_title(f"Angular Power Spectrum (Shell {shell_abs})")
ax_top.legend()
ax_top.grid(alpha=0.3)

ax_bot.axhline(1.0, color='black', linestyle='-', alpha=0.5)
ax_bot.set_ylabel(r"Ratio $C_\ell / C_\ell^{\rm theory}$")
ax_bot.set_xlabel(r"Multipole $\ell$")
ax_bot.set_ylim(0.5, 1.5)
ax_bot.grid(alpha=0.3)

plt.tight_layout()
plt.subplots_adjust(hspace=0.05)
plt.show()
